# RNN and LSTM — Exercises

Companion to the note [RNN and LSTM](RNN%20and%20LSTM.md).

Practise the recurrent update, parameter counting, why gradients vanish or explode through time, and the gating maths of LSTM and GRU cells. Then implement RNN forward, backpropagation through time, LSTM/GRU cells, gradient clipping and a bidirectional wrapper in NumPy, verified with closed forms and finite differences.

**15 exercises** · 🧠 Concept ×3 · ✍️ By hand ×5 · 💻 Code ×7

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

rng = np.random.default_rng(1)
sigmoid = lambda a: 1 / (1 + np.exp(-a))

def num_grad(f, x, eps=1e-6):
    """Central finite-difference gradient of a scalar function f() w.r.t. array x (modified in place, restored)."""
    g = np.zeros_like(x, dtype=float)
    it = np.nditer(x, flags=['multi_index'])
    for _ in it:
        i = it.multi_index
        old = x[i]
        x[i] = old + eps; fp = f()
        x[i] = old - eps; fm = f()
        x[i] = old
        g[i] = (fp - fm) / (2 * eps)
    return g

## Part A · Concepts

### Exercise 1 · Weight sharing through time
*🧠 Concept · ★☆☆*

In $h_t=\phi(W_hh_{t-1}+W_xx_t+b)$ the same $W_h, W_x, b$ are used at every step.

1. Why does the parameter count not depend on the sequence length $T$?
2. What is the analogy with weight sharing in a [[CNN]]?
3. What does "backpropagation through time" mean concretely for the gradient of $W_h$?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Unroll the network over $T$ steps: it becomes a $T$-layer feed-forward net whose layers share weights.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. The same matrices are applied at each step, so a model trained on length-20 sequences can run on length 1000.
2. A CNN shares a kernel across **space** (translation equivariance); an RNN shares weights across **time**.
3. Unroll the graph over $T$ steps and run ordinary [[Backpropagation]]. Since $W_h$ appears in every step, its gradient is the
   **sum** of the per-step contributions: $\frac{\partial L}{\partial W_h} = \sum_t \delta_t h_{t-1}^\top$ with $\delta_t$ the gradient at the pre-activation of step $t$.

</details>

### Exercise 2 · Why the LSTM fixes vanishing gradients
*🧠 Concept · ★★☆*

Explain, using the cell update $c_t=f_t\odot c_{t-1}+i_t\odot\tilde c_t$, why gradients can flow over many steps in an LSTM but not in a vanilla RNN.
Does an LSTM also solve *exploding* gradients? What is the standard remedy?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Compare $\partial h_t/\partial h_{t-1}$ of an RNN with $\partial c_t/\partial c_{t-1}$ of an LSTM.

</details>

<details>
<summary><b>✅ Solution</b></summary>

RNN: $\frac{\partial h_t}{\partial h_{t-1}} = \mathrm{diag}(\phi'(a_t))W_h$, a repeated **matrix product** that shrinks or grows geometrically.
LSTM: along the cell path $\frac{\partial c_t}{\partial c_{t-1}} = \mathrm{diag}(f_t)$ (plus indirect terms through the gates):
an **additive**, elementwise path (the "constant error carousel"). If the network learns $f_t\approx 1$, gradients pass almost unchanged.
It largely fixes *vanishing* gradients but not *exploding* ones (the indirect paths through $h$ can still blow up);
the standard remedy is **gradient clipping** by global norm (Exercise 14). A common trick is to initialise the forget bias to 1.

</details>

### Exercise 3 · GRU, bidirectional, and when to still use RNNs
*🧠 Concept · ★☆☆*

1. Name the gates of a GRU and say what replaced the LSTM's separate cell state.
2. Why can a bidirectional RNN *not* be used for real-time speech recognition with no latency, or for next-token generation?
3. The note says RNNs were largely replaced by [[Transformers]]. Give two reasons, and one setting where an RNN is still attractive.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

A bidirectional RNN reads the sequence from the end too. Transformers process all positions in parallel.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. **Update gate** $z_t$ and **reset gate** $r_t$; the GRU merges cell and hidden state into one $h_t$, with $z_t$ interpolating between $h_{t-1}$ and the candidate.
2. The backward direction needs **future** inputs, which are not available online / during autoregressive generation.
3. Transformers parallelise over time (RNNs are sequential, so slow to train on GPUs) and model long-range dependencies directly
   (path length 1 instead of $T$). RNNs remain attractive for **streaming** / on-device inference: constant memory and $O(1)$ cost per new step, no growing KV-cache.

</details>

## Part B · By hand

### Exercise 4 · Unrolling a scalar RNN
*✍️ By hand · ★☆☆*

Scalar RNN $h_t = \tanh(0.5\,h_{t-1} + x_t)$, $h_0 = 0$, inputs $x = (1, 0, -1)$. Compute $h_1, h_2, h_3$ (4 decimals).

In [ ]:
h1 = None
h2 = None
h3 = None

_h, _hs = 0.0, []
for _x in (1, 0, -1):
    _h = np.tanh(0.5 * _h + _x); _hs.append(_h)
check('h1', h1, _hs[0], tol=1e-3); check('h2', h2, _hs[1], tol=1e-3); check('h3', h3, _hs[2], tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

$h_1 = \tanh(1)$. Then $h_2 = \tanh(0.5 h_1)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$h_1=\tanh(1)=0.7616$, $h_2=\tanh(0.3808)=0.3634$, $h_3=\tanh(0.1817-1)=\tanh(-0.8183)=-0.6741$.

```python
h1, h2, h3 = 0.7616, 0.3634, -0.6741
```

</details>

### Exercise 5 · Counting parameters
*✍️ By hand · ★☆☆*

Input dimension $d=10$, hidden size $H=20$. Count the parameters (weights and biases) of one layer of

(a) a vanilla RNN, (b) an LSTM (4 blocks: $f, i, o, \tilde c$), (c) a GRU (3 blocks).

In [ ]:
p_rnn = None
p_lstm = None
p_gru = None

_blk = lambda d, H: H * d + H * H + H
check('RNN', p_rnn, _blk(10, 20)); check('LSTM', p_lstm, 4 * _blk(10, 20)); check('GRU', p_gru, 3 * _blk(10, 20))

<details>
<summary><b>💡 Hint</b></summary>

One block has $W_x \in \mathbb R^{H\times d}$, $W_h \in \mathbb R^{H\times H}$, $b\in\mathbb R^H$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

One block: $20\cdot10 + 20\cdot20 + 20 = 620$. RNN $620$, LSTM $4\cdot620 = 2480$, GRU $3\cdot620=1860$.
(PyTorch stores two bias vectors per block, so it reports $H$ more per block.)

```python
p_rnn, p_lstm, p_gru = 620, 2480, 1860
```

</details>

### Exercise 6 · Vanishing and exploding in one line
*✍️ By hand · ★☆☆*

For the linear scalar RNN $h_t = w\,h_{t-1} + x_t$, show that $\partial h_T/\partial h_0 = w^T$.
Evaluate it for $T=50$ with $w=0.9$ and with $w=1.1$. For a matrix $W_h$, what quantity plays the role of $|w|$?

In [ ]:
grad_09 = None
grad_11 = None

check('w=0.9', grad_09, 0.9 ** 50, tol=1e-4)
check('w=1.1', grad_11, 1.1 ** 50, tol=0.1)

<details>
<summary><b>💡 Hint</b></summary>

Each step multiplies by $\partial h_t/\partial h_{t-1} = w$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

By the chain rule $\partial h_T/\partial h_0 = \prod_{t=1}^T w = w^T$: $0.9^{50}\approx 0.00515$ (vanishes), $1.1^{50}\approx 117.4$ (explodes).
For matrices the gradient is $W_h^T$ (times activation derivatives), governed by the **spectral radius** / largest singular value of $W_h$.

```python
grad_09 = 0.9 ** 50
grad_11 = 1.1 ** 50
```

</details>

### Exercise 7 · An LSTM step by hand
*✍️ By hand · ★☆☆*

A scalar LSTM has gate values $f_t=0.9$, $i_t=0.2$, $o_t=0.8$, candidate $\tilde c_t=0.5$ and previous cell $c_{t-1}=1$.
Compute $c_t$ and $h_t$.

In [ ]:
c_t = None
h_t = None

check('c_t', c_t, 0.9 * 1 + 0.2 * 0.5)
check('h_t', h_t, 0.8 * np.tanh(0.9 * 1 + 0.2 * 0.5), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

$c_t=f_t c_{t-1}+i_t\tilde c_t$, $h_t=o_t\tanh c_t$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$c_t = 0.9 + 0.1 = 1.0$, $h_t = 0.8\tanh(1) = 0.8\cdot0.7616 = 0.6093$.

```python
c_t = 1.0
h_t = 0.8 * np.tanh(1.0)
```

</details>

### Exercise 8 · The constant error carousel, quantified
*✍️ By hand · ★★☆*

Along the cell path, $\partial c_T/\partial c_0 = \prod_t f_t$. Suppose the forget gate sits at $f_t = 0.99$ for 100 steps.
Compare with a tanh RNN where each step contributes a factor $|w\,\phi'| = 0.9$. Compute both gradient factors.
How large would the forget gate need to be to keep at least half of the gradient over 1000 steps?

In [ ]:
lstm_factor = None
rnn_factor = None
f_needed = None

check('LSTM factor', lstm_factor, 0.99 ** 100, tol=1e-4)
check('RNN factor', rnn_factor, 0.9 ** 100, tol=1e-7)
check('forget gate for 1000 steps', f_needed, 0.5 ** (1 / 1000), tol=1e-5)

<details>
<summary><b>💡 Hint</b></summary>

Solve $f^{1000} = 0.5$ for $f$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

LSTM: $0.99^{100}\approx0.366$. RNN: $0.9^{100}\approx 2.66\times10^{-5}$.
$f^{1000}=0.5 \Rightarrow f = 0.5^{1/1000} = e^{-\ln 2/1000}\approx 0.99931$, i.e. a forget-gate pre-activation of about $\mathrm{logit}(0.99931)\approx 7.3$.

```python
lstm_factor = 0.99 ** 100
rnn_factor = 0.9 ** 100
f_needed = 0.5 ** (1 / 1000)
```

</details>

## Part C · Code from scratch

### Exercise 9 · Vanilla RNN forward
*💻 Code · ★☆☆*

Implement `rnn_forward(X, h0, Wx, Wh, b, phi=np.tanh)` for a sequence `X` of shape `(T, d)`, returning all hidden states
as an array of shape `(T, H)`. Note the convention $h_t=\phi(W_hh_{t-1}+W_xx_t+b)$ with $W_x\in\mathbb R^{H\times d}$.

In [ ]:
def rnn_forward(X, h0, Wx, Wh, b, phi=np.tanh):
    # TODO
    return None

T_, d_, H_ = 6, 3, 4
X_r = rng.normal(size=(T_, d_)); h0_r = rng.normal(size=H_)
Wx_r = rng.normal(0, 0.5, size=(H_, d_)); Wh_r = rng.normal(0, 0.5, size=(H_, H_)); b_r = rng.normal(0, 0.1, size=H_)
H_lin = rnn_forward(X_r, h0_r, Wx_r, Wh_r, b_r, phi=lambda a: a)
H_scalar = rnn_forward(np.array([[1.], [0.], [-1.]]), np.zeros(1), np.eye(1), 0.5 * np.eye(1), np.zeros(1))

# closed form for a linear RNN: h_T = Wh^T h0 + sum_t Wh^(T-t) (Wx x_t + b)
_mp = np.linalg.matrix_power
_hT = _mp(Wh_r, T_) @ h0_r + sum(_mp(Wh_r, T_ - t) @ (Wx_r @ X_r[t - 1] + b_r) for t in range(1, T_ + 1))
check('linear RNN = closed form', None if H_lin is None else H_lin[-1], _hT)
check('scalar tanh RNN = Exercise 4', None if H_scalar is None else H_scalar[:, 0], [np.tanh(1), np.tanh(0.5*np.tanh(1)), np.tanh(0.5*np.tanh(0.5*np.tanh(1)) - 1)])

<details>
<summary><b>💡 Hint</b></summary>

Loop over $t$, keep `h`, append to a list, `np.array(list)` at the end.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The recurrence is inherently sequential: step $t$ needs $h_{t-1}$. This is what prevents parallelism over time.

```python
def rnn_forward(X, h0, Wx, Wh, b, phi=np.tanh):
    h, out = h0, []
    for x in X:
        h = phi(Wh @ h + Wx @ x + b)
        out.append(h)
    return np.array(out)

T_, d_, H_ = 6, 3, 4
X_r = rng.normal(size=(T_, d_)); h0_r = rng.normal(size=H_)
Wx_r = rng.normal(0, 0.5, size=(H_, d_)); Wh_r = rng.normal(0, 0.5, size=(H_, H_)); b_r = rng.normal(0, 0.1, size=H_)
H_lin = rnn_forward(X_r, h0_r, Wx_r, Wh_r, b_r, phi=lambda a: a)
H_scalar = rnn_forward(np.array([[1.], [0.], [-1.]]), np.zeros(1), np.eye(1), 0.5 * np.eye(1), np.zeros(1))
```

</details>

### Exercise 10 · Backpropagation through time
*💻 Code · ★★★*

For the tanh RNN of Exercise 9 and loss $L = v^\top h_T$, implement `rnn_bptt(X, h0, Wx, Wh, b, v)` returning
`(dWx, dWh, db, dh0)`. Verify against finite differences.

In [ ]:
def rnn_bptt(X, h0, Wx, Wh, b, v):
    # TODO: forward pass storing h_t, then loop t = T..1 backwards
    return None, None, None, None

v_r = rng.normal(size=H_)
dWx_r, dWh_r, db_r, dh0_r = rnn_bptt(X_r, h0_r, Wx_r, Wh_r, b_r, v_r)

def _loss():
    h = h0_r
    for x in X_r:
        h = np.tanh(Wh_r @ h + Wx_r @ x + b_r)
    return v_r @ h
check('dWx', dWx_r, num_grad(_loss, Wx_r), tol=1e-5)
check('dWh', dWh_r, num_grad(_loss, Wh_r), tol=1e-5)
check('db', db_r, num_grad(_loss, b_r), tol=1e-5)
check('dh0', dh0_r, num_grad(_loss, h0_r), tol=1e-5)

<details>
<summary><b>💡 Hint 1</b></summary>

Start with $g = \partial L/\partial h_T = v$. At step $t$: $\delta_t = g\odot(1-h_t^2)$ is the gradient at the pre-activation.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Accumulate $dW_x \mathrel{+}= \delta_t x_t^\top$, $dW_h \mathrel{+}= \delta_t h_{t-1}^\top$, $db \mathrel{+}= \delta_t$, then pass back $g = W_h^\top\delta_t$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\delta_t = g_t \odot (1-h_t^2)$, $g_{t-1} = W_h^\top \delta_t$; gradients of shared weights are **sums** over time.
After the loop, $g_0 = \partial L/\partial h_0$. Note the repeated multiplication by $W_h^\top\,\mathrm{diag}(1-h^2)$: the source of vanishing/exploding gradients.

```python
def rnn_bptt(X, h0, Wx, Wh, b, v):
    hs = [h0]
    for x in X:
        hs.append(np.tanh(Wh @ hs[-1] + Wx @ x + b))
    dWx, dWh, db = np.zeros_like(Wx), np.zeros_like(Wh), np.zeros_like(b)
    g = v.copy()
    for t in range(len(X), 0, -1):
        delta = g * (1 - hs[t] ** 2)
        dWx += np.outer(delta, X[t - 1]); dWh += np.outer(delta, hs[t - 1]); db += delta
        g = Wh.T @ delta
    return dWx, dWh, db, g

v_r = rng.normal(size=H_)
dWx_r, dWh_r, db_r, dh0_r = rnn_bptt(X_r, h0_r, Wx_r, Wh_r, b_r, v_r)
```

</details>

### Exercise 11 · Watching gradients vanish and explode
*💻 Code · ★★☆*

Let $W_h = s\,Q$ with $Q$ a random $8\times8$ orthogonal matrix (so all singular values equal $s$). For a **linear** RNN the Jacobian
$\partial h_T/\partial h_0 = W_h^T$. Compute its spectral norm for $T=1,\dots,40$ and $s\in\{0.8, 1.0, 1.2\}$; store three lists
`norms[s]`. Then explain what a tanh non-linearity would change for $s=1.2$.

In [ ]:
Q_orth, _ = np.linalg.qr(rng.normal(size=(8, 8)))
norms = None   # dict {0.8: [...], 1.0: [...], 1.2: [...]} with 40 entries each

check('s=0.8 decays like 0.8^T', None if norms is None else norms[0.8], 0.8 ** np.arange(1, 41), tol=1e-6)
check('s=1.0 stays at 1', None if norms is None else norms[1.0], np.ones(40), tol=1e-6)
check('s=1.2 grows like 1.2^T', None if norms is None else norms[1.2], 1.2 ** np.arange(1, 41), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

`np.linalg.norm(np.linalg.matrix_power(W, T), 2)` is the spectral norm. Orthogonal matrices preserve norms.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The norms are exactly $s^T$: $0.8^{40}\approx 1.3\times10^{-4}$, $1.2^{40}\approx 1470$. Orthogonal initialisation ($s=1$) is the sweet spot,
but only for a linear net. With tanh, $\phi'\le 1$ shrinks each factor, and once units saturate ($\phi'\approx 0$) the gradient vanishes even for $s>1$;
exploding gradients then appear only in parts of state space, which is why clipping is used rather than a fixed rescaling.

```python
Q_orth, _ = np.linalg.qr(rng.normal(size=(8, 8)))
norms = {s: [np.linalg.norm(np.linalg.matrix_power(s * Q_orth, T), 2) for T in range(1, 41)] for s in (0.8, 1.0, 1.2)}
print({s: round(v[-1], 6) for s, v in norms.items()})
```

</details>

### Exercise 12 · LSTM cell and sequence forward
*💻 Code · ★★☆*

Implement `lstm_step(x, h, c, W, b)` where `W` has shape `(4H, d+H)` and acts on the concatenation $[x; h]$,
with blocks in the order $(f, i, o, \tilde c)$: $f,i,o = \sigma(\cdot)$, $\tilde c = \tanh(\cdot)$. Return `(h_new, c_new)`.
Then `lstm_forward(X, h0, c0, W, b)` returns the list of all `h` and the final `c`.

In [ ]:
def lstm_step(x, h, c, W, b):
    return None, None

def lstm_forward(X, h0, c0, W, b):
    return None, None

d_l, H_l = 3, 2
X_l = rng.normal(size=(10, d_l))
# test 1: zero weights, biases chosen so f=0.9, i=0.5, o=0.5, c~=0.5
b_test = np.r_[np.full(H_l, np.log(9)), np.zeros(H_l), np.zeros(H_l), np.full(H_l, np.arctanh(0.5))]
hs_1, cT_1 = lstm_forward(X_l, np.zeros(H_l), np.ones(H_l), np.zeros((4 * H_l, d_l + H_l)), b_test)
# test 2: random weights but saturated gates f=1, i=0  -> the cell must remember c0
W_rand = rng.normal(size=(4 * H_l, d_l + H_l))
b_mem = np.r_[np.full(H_l, 50.), np.full(H_l, -50.), np.zeros(H_l), np.zeros(H_l)]
c0_mem = np.array([0.7, -1.3])
hs_2, cT_2 = lstm_forward(X_l, np.zeros(H_l), c0_mem, W_rand, b_mem)

# closed form of test 1: c_T = 0.9^T c0 + 0.25 (1-0.9^T)/0.1
_cT = 0.9 ** 10 * 1 + 0.25 * (1 - 0.9 ** 10) / 0.1
check('constant gates: c_T closed form', cT_1, np.full(2, _cT))
check('constant gates: h_T = 0.5 tanh(c_T)', None if hs_1 is None else hs_1[-1], np.full(2, 0.5 * np.tanh(_cT)))
check('f=1, i=0: cell remembers c0 for 10 steps', cT_2, c0_mem)

<details>
<summary><b>💡 Hint</b></summary>

`z = W @ np.r_[x, h] + b`; `f, i, o, g = np.split(z, 4)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Test 1 is a geometric recursion $c_t = 0.9c_{t-1} + 0.25$ with fixed point $2.5$. Test 2 shows the memory mechanism: with $f=1, i=0$
the cell state is copied unchanged whatever the inputs, which is exactly the gradient highway of Exercise 2.

```python
def lstm_step(x, h, c, W, b):
    f, i, o, g = np.split(W @ np.r_[x, h] + b, 4)
    f, i, o, g = sigmoid(f), sigmoid(i), sigmoid(o), np.tanh(g)
    c_new = f * c + i * g
    return o * np.tanh(c_new), c_new

def lstm_forward(X, h0, c0, W, b):
    h, c, hs = h0, c0, []
    for x in X:
        h, c = lstm_step(x, h, c, W, b)
        hs.append(h)
    return np.array(hs), c

d_l, H_l = 3, 2
X_l = rng.normal(size=(10, d_l))
b_test = np.r_[np.full(H_l, np.log(9)), np.zeros(H_l), np.zeros(H_l), np.full(H_l, np.arctanh(0.5))]
hs_1, cT_1 = lstm_forward(X_l, np.zeros(H_l), np.ones(H_l), np.zeros((4 * H_l, d_l + H_l)), b_test)
W_rand = rng.normal(size=(4 * H_l, d_l + H_l))
b_mem = np.r_[np.full(H_l, 50.), np.full(H_l, -50.), np.zeros(H_l), np.zeros(H_l)]
c0_mem = np.array([0.7, -1.3])
hs_2, cT_2 = lstm_forward(X_l, np.zeros(H_l), c0_mem, W_rand, b_mem)
```

</details>

### Exercise 13 · GRU cell
*💻 Code · ★★☆*

Implement a GRU step (d2l convention):
$z=\sigma(W_z[x;h]+b_z)$, $r=\sigma(W_r[x;h]+b_r)$, $\tilde h=\tanh(W_c[x;\,r\odot h]+b_c)$, $h'=z\odot h+(1-z)\odot\tilde h$.
`P` is a dict with keys `Wz, bz, Wr, br, Wc, bc`.

In [ ]:
def gru_step(x, h, P):
    return None

d_g, H_g = 3, 4
x_g, h_g = rng.normal(size=d_g), rng.normal(size=H_g)
P = {k: rng.normal(size=(H_g, d_g + H_g)) for k in ('Wz', 'Wr', 'Wc')}
P.update({k: np.zeros(H_g) for k in ('bz', 'br', 'bc')})
P_keep = dict(P, bz=np.full(H_g, 50.))                      # z = 1
P_reset = dict(P, bz=np.full(H_g, -50.), br=np.full(H_g, -50.))   # z = 0, r = 0
h_keep = gru_step(x_g, h_g, P_keep)
h_reset = gru_step(x_g, h_g, P_reset)

check('z=1: state is copied', h_keep, h_g)
check('z=0, r=0: h = tanh(Wc[:, :d] x + bc)', h_reset, np.tanh(P['Wc'][:, :d_g] @ x_g))

<details>
<summary><b>💡 Hint</b></summary>

Compute `z` and `r` from `np.r_[x, h]`, then the candidate from `np.r_[x, r * h]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

With $z=1$ the GRU copies its state (like $f=1, i=0$ in the LSTM); with $r=0$ the candidate ignores the past entirely.
The GRU couples "forget" and "input" via $z$ and $1-z$, which is why it needs one gate fewer.

```python
def gru_step(x, h, P):
    xh = np.r_[x, h]
    z = sigmoid(P['Wz'] @ xh + P['bz'])
    r = sigmoid(P['Wr'] @ xh + P['br'])
    h_tilde = np.tanh(P['Wc'] @ np.r_[x, r * h] + P['bc'])
    return z * h + (1 - z) * h_tilde

d_g, H_g = 3, 4
x_g, h_g = rng.normal(size=d_g), rng.normal(size=H_g)
P = {k: rng.normal(size=(H_g, d_g + H_g)) for k in ('Wz', 'Wr', 'Wc')}
P.update({k: np.zeros(H_g) for k in ('bz', 'br', 'bc')})
P_keep = dict(P, bz=np.full(H_g, 50.))
P_reset = dict(P, bz=np.full(H_g, -50.), br=np.full(H_g, -50.))
h_keep = gru_step(x_g, h_g, P_keep)
h_reset = gru_step(x_g, h_g, P_reset)
```

</details>

### Exercise 14 · Gradient clipping by global norm
*💻 Code · ★☆☆*

Implement `clip_global(grads, max_norm)`: if the global norm $\sqrt{\sum_k \lVert g_k\rVert^2}$ exceeds `max_norm`,
scale **all** gradients by `max_norm / norm`; otherwise return them unchanged. Why scale globally rather than clip each element?

In [ ]:
def clip_global(grads, max_norm):
    return None

g_big = [np.array([3.0, 4.0]), np.array([[12.0]])]      # global norm 13
g_small = [np.array([0.3, 0.4])]
clipped_big = clip_global(g_big, 1.0)
clipped_small = clip_global(g_small, 1.0)

_n = lambda gs: np.sqrt(sum(np.sum(g ** 2) for g in gs))
check('clipped norm = max_norm', None if clipped_big is None else _n(clipped_big), 1.0)
check('direction preserved', None if clipped_big is None else np.r_[clipped_big[0], clipped_big[1].ravel()], np.array([3, 4, 12.]) / 13)
check('small gradients unchanged', None if clipped_small is None else clipped_small[0], g_small[0])

<details>
<summary><b>💡 Hint</b></summary>

Compute the norm once over all arrays, then `[g * scale for g in grads]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Global-norm clipping rescales the whole update, preserving its **direction** (it is still a descent direction); elementwise clipping
changes the direction. It is standard for RNNs and Transformers ([[Training Tricks]]).

```python
def clip_global(grads, max_norm):
    norm = np.sqrt(sum(np.sum(g ** 2) for g in grads))
    scale = min(1.0, max_norm / (norm + 1e-12))
    return [g * scale for g in grads]

g_big = [np.array([3.0, 4.0]), np.array([[12.0]])]
g_small = [np.array([0.3, 0.4])]
clipped_big = clip_global(g_big, 1.0)
clipped_small = clip_global(g_small, 1.0)
```

</details>

### Exercise 15 · Bidirectional RNN
*💻 Code · ★★☆*

Using your `rnn_forward`, implement `birnn(X, fwd, bwd)` where `fwd` and `bwd` are tuples `(h0, Wx, Wh, b)`.
Run one RNN left-to-right and another right-to-left, **re-align** the backward states with the original time index, and
concatenate: output shape `(T, 2H)`. Which position's output has seen the whole sequence in *both* directions?

In [ ]:
def birnn(X, fwd, bwd):
    return None

fwd_p = (np.zeros(H_), Wx_r, Wh_r, b_r)
bwd_p = (np.zeros(H_), rng.normal(0, 0.5, size=(H_, d_)), rng.normal(0, 0.5, size=(H_, H_)), np.zeros(H_))
H_bi = birnn(X_r, fwd_p, bwd_p)

def _run(X, h0, Wx, Wh, b):
    h, out = h0, []
    for x in X:
        h = np.tanh(Wh @ h + Wx @ x + b); out.append(h)
    return np.array(out)
_ref = np.c_[_run(X_r, *fwd_p), _run(X_r[::-1], *bwd_p)[::-1]]
check('bidirectional states', H_bi, _ref)

<details>
<summary><b>💡 Hint</b></summary>

Run the backward RNN on `X[::-1]` and reverse its output again with `[::-1]` before concatenating.

</details>

<details>
<summary><b>✅ Solution</b></summary>

At every position $t$, the concatenation $[\overrightarrow{h}_t; \overleftarrow{h}_t]$ summarises the **past** (forward) and the **future**
(backward), so each position's output depends on the whole sequence, not only the last one. That is ideal for tagging (NER, POS) but impossible online.

```python
def birnn(X, fwd, bwd):
    Hf = rnn_forward(X, *fwd)
    Hb = rnn_forward(X[::-1], *bwd)[::-1]
    return np.concatenate([Hf, Hb], axis=1)

fwd_p = (np.zeros(H_), Wx_r, Wh_r, b_r)
bwd_p = (np.zeros(H_), rng.normal(0, 0.5, size=(H_, d_)), rng.normal(0, 0.5, size=(H_, H_)), np.zeros(H_))
H_bi = birnn(X_r, fwd_p, bwd_p)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [RNN and LSTM](RNN%20and%20LSTM.md).*